# 07. Pretrained audio models vs. recording time on D2 (E7)

- Encoders (frozen): wav2vec 2.0, HuBERT, AST; in-house 2023 mel features (feature vector, not CNN)
- Probe: logistic regression (all representations)
- Audio: first 10 s per file
- Populations: 140 WAV (all audio files), 118 timed WAV, 70 files in 03:30–04:30

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation as ev, sfb, reporting as rp
cfg.apply_mpl_style(); cfg.banner()
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier
import warnings; warnings.filterwarnings("ignore")

M2 = io_utils.scan_d2()
D2W = M2[(M2.ext == ".wav") & (M2.label.isin(["leak", "noleak"]))].reset_index(drop=True)
D2W["logger"] = D2W.logger.fillna("NA")
print(f"D2 wav: {len(D2W)}   loggers: {D2W.logger.nunique()}   timed: {int(D2W.timed.sum())}")

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000
D2 wav: 140   loggers: 50   timed: 118


## 1. Feature extractors

- Mel2023: band-pass → min-std 1 s window → summary features
- wav2vec 2.0, HuBERT: torchaudio pipelines (WAV2VEC2_BASE, HUBERT_BASE), resampled to model rate
- AST: MIT/ast-finetuned-audioset-10-10-0.4593, resampled to 16 kHz
- Embedding: mean + SD of last-layer hidden states over time

In [3]:
EXTRACTORS = {}

def embed_mel2023(x, sr):
    xb = features.bandpass(x, sr)
    xw = features.pick_window(xb, sr, sec=cfg.WIN_SEC, mode="minstd")
    return features.summary_features(xw, sr)
EXTRACTORS["Mel2023"] = embed_mel2023

NAMES = ["Mel2023", "WAV2VEC2_BASE", "HUBERT_BASE", "AST"]
need = [n for n in NAMES if io_utils.load_cache(f"d2_emb_{n}", D2W.file_id) is None]
USE_FOUNDATION = bool(set(need) - {"Mel2023"})
if USE_FOUNDATION:
    try:
        import torch, librosa
        DEV = "cuda" if torch.cuda.is_available() else "cpu"
        for bundle_name in ["WAV2VEC2_BASE", "HUBERT_BASE"]:
            try:
                import torchaudio
                bundle = getattr(torchaudio.pipelines, bundle_name)
                model = bundle.get_model().to(DEV).eval()
                sr_req = bundle.sample_rate
                def make_fn(m, srq):
                    def f(x, sr):
                        xr = librosa.resample(x, orig_sr=sr, target_sr=srq) if sr != srq else x
                        with torch.no_grad():
                            t = torch.tensor(xr, dtype=torch.float32, device=DEV)[None]
                            feats, _ = m.extract_features(t)
                            h = feats[-1]
                            return torch.cat([h.mean(1), h.std(1)], -1).cpu().numpy().ravel()
                    return f
                EXTRACTORS[bundle_name] = make_fn(model, sr_req)
                print(f"  {bundle_name} ready")
            except Exception as e:
                print(f"  {bundle_name} unavailable: {str(e)[:80]}")
        try:
            from transformers import ASTFeatureExtractor, ASTModel
            _fe = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            _md = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593").to(DEV).eval()
            def embed_ast(x, sr):
                xr = librosa.resample(x, orig_sr=sr, target_sr=16000) if sr != 16000 else x
                with torch.no_grad():
                    inp = _fe(xr, sampling_rate=16000, return_tensors="pt")
                    h = _md(**{k: v.to(DEV) for k, v in inp.items()}).last_hidden_state
                    return torch.cat([h.mean(1), h.std(1)], -1).cpu().numpy().ravel()
            EXTRACTORS["AST"] = embed_ast
            print("  AST ready")
        except Exception as e:
            print(f"  AST unavailable: {str(e)[:80]}")
    except ImportError:
        print("  torch/torchaudio not installed - baseline only")
print("caches missing:", need)

  WAV2VEC2_BASE ready
  HUBERT_BASE ready


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  AST ready
caches missing: ['Mel2023', 'WAV2VEC2_BASE', 'HUBERT_BASE', 'AST']


## 2. Embedding extraction (cached with file IDs)

In [4]:
def embed_all(name, fn, max_sec=cfg.D2_MAX_SEC):
    c = io_utils.load_cache(f"d2_emb_{name}", D2W.file_id)
    if c is not None:
        return c["X"]
    if fn is None:
        return None
    import librosa
    Xs = []
    for p in tqdm(D2W.path, desc=name):
        x, sr = librosa.load(p, sr=None, mono=True, duration=max_sec)
        Xs.append(fn(x, sr))
    X = np.vstack(Xs)
    io_utils.save_cache(f"d2_emb_{name}", X, D2W.file_id)
    return X

EMB = {}
for name in NAMES:
    X = embed_all(name, EXTRACTORS.get(name))
    if X is not None:
        EMB[name] = X
for k, v in EMB.items():
    print(f"  {k:14s} {v.shape}")

Mel2023:   0%|          | 0/140 [00:00<?, ?it/s]

WAV2VEC2_BASE:   0%|          | 0/140 [00:00<?, ?it/s]

HUBERT_BASE:   0%|          | 0/140 [00:00<?, ?it/s]

AST:   0%|          | 0/140 [00:00<?, ?it/s]

  Mel2023        (140, 199)
  WAV2VEC2_BASE  (140, 1536)
  HUBERT_BASE    (140, 1536)
  AST            (140, 1536)


## 3. E7 (v1 design): naive vs. logger-independent

- Populations: 140 WAV, 118 timed WAV
- P1 naive: stratified 70/30 / P2 logger-independent: GroupShuffleSplit by logger, 35% test
- 10 seeds; P2 draws without both classes skipped

In [5]:
y2, lg2 = D2W.label.to_numpy(), D2W.logger.to_numpy()

def e7_design(rows_idx, tag):
    rows = []
    yy, gg = y2[rows_idx], lg2[rows_idx]
    for name, Xe in EMB.items():
        Xs = Xe[rows_idx]
        for seed in range(cfg.N_SEEDS):
            a, b = train_test_split(np.arange(len(yy)), test_size=.3, random_state=seed, stratify=yy)
            r = ev.evaluate(Xs[a], yy[a], Xs[b], yy[b], "logreg", seed)
            rows.append(dict(population=tag, model=name, protocol="P1_naive", seed=seed,
                             **{k: r[k] for k in ["acc", "bacc", "mf1", "base"]}))
            a, b = next(GroupShuffleSplit(1, test_size=.35, random_state=seed)
                        .split(np.arange(len(yy)), yy, groups=gg))
            if len(np.unique(yy[a])) > 1 and len(np.unique(yy[b])) > 1:
                r = ev.evaluate(Xs[a], yy[a], Xs[b], yy[b], "logreg", seed)
                rows.append(dict(population=tag, model=name, protocol="P2_logger_indep", seed=seed,
                                 **{k: r[k] for k in ["acc", "bacc", "mf1", "base"]}))
    return rows

rows = e7_design(np.arange(len(D2W)), "wav140") + e7_design(np.where(D2W.timed)[0], "timed118")
e7 = pd.DataFrame(rows)
e7.to_csv(cfg.RESULTS / "E7_foundation_raw.csv", index=False)
s7 = e7.groupby(["population", "model", "protocol"]).agg(
    acc=("acc", "mean"), sd=("acc", "std"), bacc=("bacc", "mean"),
    base=("base", "mean"), n=("acc", "size")).reset_index()
s7.to_csv(cfg.RESULTS / "E7_foundation_summary.csv", index=False)
print(s7.round(4).to_string(index=False))
for _, r in s7.iterrows():
    rp.record(f"D2.E7.{r.population}.{r.model}.{r.protocol}.acc", r.acc, "E7_foundation_summary.csv",
              "R1-M7", note=f"sd over seeds={r.sd:.4f}, bacc={r.bacc:.4f}")

population         model        protocol    acc     sd   bacc   base  n
  timed118           AST        P1_naive 0.9250 0.0525 0.9167 0.5833 10
  timed118           AST P2_logger_indep 0.8318 0.0989 0.8228 0.5923 10
  timed118   HUBERT_BASE        P1_naive 0.9083 0.0372 0.8995 0.5833 10
  timed118   HUBERT_BASE P2_logger_indep 0.7652 0.1003 0.7523 0.5923 10
  timed118       Mel2023        P1_naive 0.8556 0.0537 0.8419 0.5833 10
  timed118       Mel2023 P2_logger_indep 0.7916 0.1249 0.7811 0.5923 10
  timed118 WAV2VEC2_BASE        P1_naive 0.8972 0.0473 0.8900 0.5833 10
  timed118 WAV2VEC2_BASE P2_logger_indep 0.7937 0.0706 0.7824 0.5923 10
    wav140           AST        P1_naive 0.8619 0.0386 0.8619 0.5000 10
    wav140           AST P2_logger_indep 0.7665 0.1175 0.7819 0.6078 10
    wav140   HUBERT_BASE        P1_naive 0.8500 0.0539 0.8500 0.5000 10
    wav140   HUBERT_BASE P2_logger_indep 0.7885 0.1161 0.8109 0.6078 10
    wav140       Mel2023        P1_naive 0.7333 0.0708 0.7333 0.

## 4. Acoustic vs. clock (same files, same folds)

- Folds: StratifiedGroupKFold by logger (5 folds), 10 fold seeds
- Predictors: clock (depth-1 stump on hour), each embedding, embedding + standardised hour
- Populations: 118 timed WAV, 70 files in 03:30–04:30
- Intervals (fold seed 0): logger-level bootstrap for accuracy; paired logger-level bootstrap for differences vs. clock

In [6]:
class Stump:
    def __init__(self): self.m = DecisionTreeClassifier(max_depth=1, random_state=0)
    def fit(self, X, y): self.m.fit(X[:, -1:], y); return self
    def predict(self, X): return self.m.predict(X[:, -1:])

def oof(Xs, yy, folds, which, seed=0):
    pred = np.empty(len(yy), dtype=object)
    for tr, te in folds:
        m = Stump() if which == "clock" else ev.make_model("logreg", seed)
        pred[te] = m.fit(Xs[tr], yy[tr]).predict(Xs[te])
    return pred.astype(str)

def compare(idx, tag, n_fold_seeds=cfg.N_SEEDS):
    yy, gg = y2[idx], lg2[idx]
    hour = D2W.hour.to_numpy()[idx].reshape(-1, 1)
    hz = (hour - hour.mean()) / (hour.std() + 1e-9)
    n_splits = min(5, len(set(gg)))
    res, store = [], {}
    for fs in range(n_fold_seeds):
        folds = list(StratifiedGroupKFold(n_splits, shuffle=True, random_state=fs).split(hz, yy, gg))
        rule = ev.fold_majority_rule(yy, folds)
        sets = {"clock (hour stump)": ("clock", hz)}
        for name, Xe in EMB.items():
            sets[f"{name}"] = ("acoustic", Xe[idx])
            sets[f"{name} + clock"] = ("acoustic", np.hstack([Xe[idx], hz]))
        for sname, (which, Xs) in sets.items():
            pr = oof(Xs, yy, folds, which, seed=fs)
            ok = pr == yy
            s = ev.score(yy, pr)
            rec = dict(population=tag, fold_seed=fs, predictor=sname, n=len(yy), acc=s["acc"],
                       bacc=s["bacc"], rule=float(rule.mean()))
            if fs == 0:
                ci = sfb.cluster_bootstrap_acc(ok, gg, n_boot=cfg.N_BOOT, base_correct=rule)
                rec.update(lo=ci["lo"], hi=ci["hi"], n_loggers=ci["n_clusters"])
                store[sname] = pr
            res.append(rec)
    res = pd.DataFrame(res)
    summ = (res.groupby("predictor").agg(n=("n", "first"), acc=("acc", "mean"), acc_sd_foldseeds=("acc", "std"),
                                         bacc=("bacc", "mean"), rule=("rule", "mean"),
                                         lo_seed0=("lo", "first"), hi_seed0=("hi", "first"),
                                         n_loggers=("n_loggers", "first"))
            .reset_index().assign(population=tag))
    # paired differences against the clock (fold seed 0)
    diffs = []
    for sname in store:
        if sname == "clock (hour stump)":
            continue
        r = sfb.paired_cluster_bootstrap_metrics(yy, store[sname], store["clock (hour stump)"], gg,
                                                 n_boot=cfg.N_BOOT)
        diffs.append(dict(population=tag, comparison=f"{sname} - clock", **r))
    return summ, pd.DataFrame(diffs)

timed = np.where(D2W.timed)[0]
early = np.where(D2W.timed & (D2W.hour >= 3.5) & (D2W.hour <= 4.5))[0]
s_t, d_t = compare(timed, "timed118")
s_e, d_e = compare(early, "early_0330_0430")
avc = pd.concat([s_t, s_e], ignore_index=True)
avd = pd.concat([d_t, d_e], ignore_index=True)
avc.to_csv(cfg.RESULTS / "E7_acoustic_vs_clock.csv", index=False)
avd.to_csv(cfg.RESULTS / "E7_acoustic_vs_clock_differences.csv", index=False)
print(avc.round(4).to_string(index=False))
print()
print(avd[["population", "comparison", "d_acc", "d_acc_lo", "d_acc_hi", "d_bacc", "d_bacc_lo", "d_bacc_hi"]]
      .round(4).to_string(index=False))
for _, r in avc.iterrows():
    rp.record(f"D2.avc.{r.population}.{r.predictor}.acc", r.acc, "E7_acoustic_vs_clock.csv", "R2-3,R2-6",
              lo=r.lo_seed0, hi=r.hi_seed0, note=f"bacc={r.bacc:.4f}; interval: logger bootstrap (fold seed 0)")
for _, r in avd.iterrows():
    rp.record(f"D2.avc.{r.population}.{r.comparison}.d_acc", r.d_acc, "E7_acoustic_vs_clock_differences.csv",
              "R2-3,R2-6", lo=r.d_acc_lo, hi=r.d_acc_hi, note="unit=logger")

            predictor   n    acc  acc_sd_foldseeds   bacc   rule  lo_seed0  hi_seed0  n_loggers      population
                  AST 118 0.8619            0.0215 0.8443 0.5932    0.7478    0.9445       32.0        timed118
          AST + clock 118 0.8627            0.0199 0.8453 0.5932    0.7478    0.9445       32.0        timed118
          HUBERT_BASE 118 0.8093            0.0292 0.7823 0.5932    0.6829    0.9033       32.0        timed118
  HUBERT_BASE + clock 118 0.8127            0.0295 0.7852 0.5932    0.7000    0.9196       32.0        timed118
              Mel2023 118 0.7797            0.0155 0.7583 0.5932    0.6448    0.8761       32.0        timed118
      Mel2023 + clock 118 0.8008            0.0151 0.7781 0.5932    0.6439    0.8750       32.0        timed118
        WAV2VEC2_BASE 118 0.8271            0.0220 0.8078 0.5932    0.6860    0.9126       32.0        timed118
WAV2VEC2_BASE + clock 118 0.8339            0.0227 0.8155 0.5932    0.7079    0.9225       32.0        t